# Reproducible Reanalysis: Credit Card Fraud Detection Under Severe Class Imbalance

This notebook reproduces the journal manuscript's primary 14-feature analysis. It evaluates logistic regression, decision tree, and Gaussian naïve Bayes with and without random oversampling. ROC-AUC is calculated from predicted probabilities and PR-AUC is reported because the dataset is severely imbalanced.

**Important:** The original third-party `creditcard.csv` dataset is intentionally not included in this repository. Obtain it from its authorized public source and place it at `data/creditcard.csv`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_curve,
    precision_recall_curve
)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

RANDOM_STATE = 42
DATA_PATH = Path('data/creditcard.csv')
RESULTS_DIR = Path('results')
FIGURES_DIR = Path('figures')
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)


In [ ]:
df = pd.read_csv(DATA_PATH)
features = ['V1','V2','V3','V4','V5','V7','V9','V10','V11','V12','V14','V16','V17','V18']
X = df[features]
y = df['Class']
print(f'Transactions: {len(df):,}')
print(f'Fraudulent transactions: {int(y.sum()):,}')
print(f'Fraud rate: {100*y.mean():.4f}%')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print('Train:', X_train.shape, 'Test:', X_test.shape)


In [ ]:
def fit_and_score(name, model, oversample=False):
    if oversample:
        model = Pipeline([
            ('oversampler', RandomOverSampler(sampling_strategy=1.0, random_state=RANDOM_STATE)),
            ('model', model)
        ])
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    return {
        'Model': name,
        'Training': 'Oversampled' if oversample else 'Original',
        'ROC-AUC': roc_auc_score(y_test, prob),
        'PR-AUC': average_precision_score(y_test, prob),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'Specificity': tn/(tn+fp),
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
        'prob': prob
    }

results = []
models = {
    'Logistic Regression': LogisticRegression(C=1, max_iter=2000, solver='lbfgs', random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Naive Bayes': GaussianNB(),
}
for name, model in models.items():
    for oversample in [False, True]:
        results.append(fit_and_score(name, model, oversample))

table = pd.DataFrame([{k:v for k,v in r.items() if k != 'prob'} for r in results])
table.to_csv(RESULTS_DIR/'primary_holdout_results.csv', index=False)
table.round(4)


In [ ]:
plt.figure(figsize=(7, 5.5))
for r in results:
    fpr, tpr, _ = roc_curve(y_test, r['prob'])
    plt.plot(fpr, tpr, label=f"{r['Model']} ({r['Training']}): {r['ROC-AUC']:.3f}")
plt.plot([0,1], [0,1], '--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Corrected Holdout Analysis')
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(FIGURES_DIR/'ROC_curves.png', dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(7, 5.5))
for r in results:
    precision, recall, _ = precision_recall_curve(y_test, r['prob'])
    plt.plot(recall, precision, label=f"{r['Model']} ({r['Training']}): {r['PR-AUC']:.3f}")
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves — Corrected Holdout Analysis')
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(FIGURES_DIR/'PR_curves.png', dpi=300)
plt.show()


## Reproducibility note

The original notebook used an unseeded 80:20 split and computed the reported AUC from hard class predictions. The journal reanalysis instead uses a stratified split with `random_state=42`, applies oversampling only to the training set, and calculates ROC-AUC from predicted probabilities. This distinction is intentional and is documented in the manuscript.